> **Portfolio summary:** Generating synthetic fraud-transaction data with CTGAN and measuring its quality (SDV score 0.68, discriminator AUC 0.71).
>
> [Project README](README.md) · [Back to the portfolio](../README.md)

This notebook covers loading fraud_transactions.csv and preparing the data by handling missing values and defining target/feature columns. Real data was split into training and testing sets. Synthetic data was then generated using a CTGAN model trained for 300 epochs. Data quality was evaluated through an SDV QualityReport, yielding an overall score of 0.6846, and a discriminator AUC score of 0.7140. The goal was to generate and assess synthetic transaction data mirroring real data characteristics.

In [ ]:
!pip install sdv

In [ ]:
import pandas as pd
import numpy as np

from datasets import load_dataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score

from scipy.stats import ks_2samp
from sklearn.neighbors import NearestNeighbors

In [ ]:
df = pd.read_csv('fraud_transactions.csv')
df.head()

,trans_date_trans_time,merchant,category,amt,gender,state,job,is_fraud
0,2/27/19 21:32,"fraud_Langosh, Wintheiser and Hyatt",food_dining,83.64,F,TX,"Physicist, medical",0
1,2/13/19 19:41,fraud_Dibbert and Sons,entertainment,79.13,M,PA,Secretary/administrator,0
2,1/11/19 20:03,"fraud_McDermott, Osinski and Morar",home,12.02,F,CA,"Buyer, industrial",0
3,1/20/19 9:08,fraud_Bauch-Raynor,grocery_pos,84.41,M,TN,Clothing/textile technologist,0
4,1/4/19 17:04,"fraud_Reichert, Huels and Hoppe",shopping_net,2.81,F,ME,Financial trader,0


In [ ]:
df = df.dropna().reset_index(drop=True)

target_col = "is_fraud"

cat_cols = ["merchant", "category","gender","state","job"]
num_cols = [
    "trans_date_trans_time",
    "amt"
]

X = df[cat_cols + num_cols].copy()
y = df[target_col].astype(int)

df[[target_col]].value_counts(normalize=True)


,proportion
is_fraud,
0,0.925069
1,0.074931


In [ ]:
real_train, real_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df[target_col]
)

real_train.shape, real_test.shape


((5188, 8), (1298, 8))

In [ ]:
from ctgan import CTGAN

ctgan_data = df[cat_cols + num_cols + [target_col]].copy()

discrete_cols = cat_cols + ["trans_date_trans_time", "amt", target_col]

ctgan = CTGAN(
    epochs=300,       # increase for better quality if you have time/compute
    batch_size=500,
    verbose=True
)

ctgan.fit(ctgan_data, discrete_columns=discrete_cols)


In [ ]:

n_synth = 5000
synthetic = ctgan.sample(n_synth)
synthetic.head()


,merchant,category,gender,state,job,trans_date_trans_time,amt,is_fraud
0,"fraud_Swift, Bradtke and Marquardt",food_dining,M,MI,"Librarian, academic",2/17/19 23:52,43.10,0
1,fraud_Emmerich-Rau,misc_net,F,ND,Tax adviser,1/15/19 23:04,51.89,0
2,fraud_Cole PLC,gas_transport,M,IA,"Engineer, control and instrumentation",2/5/19 5:16,61.42,0
3,fraud_Schmeler Inc,food_dining,M,OR,"Loss adjuster, chartered",2/3/19 12:17,35.91,0
4,fraud_Hintz-Bruen,health_fitness,F,NH,"Designer, ceramics/pottery",2/11/19 19:30,180.38,0


In [ ]:
from sdmetrics.reports.single_table import QualityReport

# 1. Statistical Evaluation using SDV metrics
report = QualityReport()
report.generate(real_train, synthetic, metadata={'columns': {col: {'sdtype': 'categorical' if col in cat_cols + [target_col] else 'numerical'} for col in real_train.columns}})

print("Overall Quality Score:", report.get_score())
display(report.get_details(property_name='Column Shapes'))

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 8/8 [00:00<00:00, 74.16it/s]|
Column Shapes Score: 87.1%

(2/2) Evaluating Column Pair Trends: |██████████| 28/28 [00:00<00:00, 140.14it/s]|
Column Pair Trends Score: 49.83%

Overall Score (Average): 68.46%

Overall Quality Score: 0.684635530765535

,Column,Metric,Score
0,trans_date_trans_time,KSComplement,0.972667
1,merchant,TVComplement,0.814000
2,category,TVComplement,0.870224
3,amt,KSComplement,0.878569
4,gender,TVComplement,0.874607
5,state,TVComplement,0.776927
6,job,TVComplement,0.816134
7,is_fraud,TVComplement,0.964981


In [ ]:
# 2. Discriminator Score (Detection Accuracy)
# Combine real and synthetic data with a new label
real_data_eval = real_train.copy()
real_data_eval['is_synthetic'] = 0
synthetic_eval = synthetic.copy()
synthetic_eval['is_synthetic'] = 1

combined = pd.concat([real_data_eval, synthetic_eval], axis=0)

# Simple preprocessing for the discriminator
X_disc = pd.get_dummies(combined.drop('is_synthetic', axis=1), drop_first=True)
y_disc = combined['is_synthetic']

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(X_disc, y_disc, test_size=0.2, random_state=42)

discriminator = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
discriminator.fit(X_train_d, y_train_d)

auc = roc_auc_score(y_test_d, discriminator.predict_proba(X_test_d)[:, 1])
print(f"Discriminator AUC Score: {auc:.4f}")
print("(An AUC close to 0.50 means the synthetic data is indistinguishable from real data)")

Discriminator AUC Score: 0.7140
(An AUC close to 0.50 means the synthetic data is indistinguishable from real data)